# Ingesta de datos históricos de sequía en Sonora

Esta libreta de Jupyter se utilizará para la ingesta y limpieza de los datos históricos de sequía en Sonora. A partir de los conjuntos de datos públicos de la CONAGUA, se utilizaran las variables de intensidad, frecuencia y temporalidad a nivel municipal a lo largo de los años. El pipeline contempla el proceso y transformación de los datos para facilitar su análisis exploratorio y visualización.

## Ingesta de datos

### Librerías necesarias

In [6]:
#!pip install -q pandas requests pydantic

In [ ]:
from pathlib import Path
import pandas as pd
from loguru import logger
import requests
import sys
from pydantic import ValidationError
ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
  sys.path.append(str(ROOT_DIR))
  
from sonora_agriclimate_eda.schemas import SequiaSonoraSchema
from sonora_agriclimate_eda.dataset import download_drought_data

start_year = 2019
end_year = 2023

# 1. Define paths from the notebook
root = Path().resolve().parent
interim_dir = root / "data/interim"
output_dir = interim_dir / "sonora_sequia"

# 2. Ensure the folder exists and download directly.
output_dir.mkdir(parents=True, exist_ok=True)

conagua_url = "https://smn.conagua.gob.mx/tools/RESOURCES/Monitor de Sequia en Mexico/MunicipiosSequia.xlsx"

## Descarga de datos

In [14]:
download_drought_data(start_year, end_year)

2026-09-29 00:07:30.297 | INFO     | sonora_agriclimate_eda.dataset:download_drought_data:65 - Iniciando descarga de datos de sequía desde: https://smn.conagua.gob.mx/tools/RESOURCES/Monitor de Sequia en Mexico/MunicipiosSequia.xlsx
2026-09-29 00:07:32.151 | SUCCESS  | sonora_agriclimate_eda.dataset:download_drought_data:69 - ¡Archivo descargado y guardado con éxito en: C:\Users\patyq\OneDrive - Universidad de Sonora\MDC\IngCaracteristicas\sonora-agriclimate-eda\data\raw\MunicipiosSequia.xlsx!
2026-09-29 00:07:32.152 | INFO     | sonora_agriclimate_eda.dataset:download_drought_data:78 - Leyendo el archivo crudo de CONAGUA desde C:\Users\patyq\OneDrive - Universidad de Sonora\MDC\IngCaracteristicas\sonora-agriclimate-eda\data\raw\MunicipiosSequia.xlsx...
2026-09-29 00:07:37.390 | INFO     | sonora_agriclimate_eda.dataset:download_drought_data:105 - Transformando a formato largo (Tidy Data)...
2026-09-29 00:07:37.511 | SUCCESS  | sonora_agriclimate_eda.dataset:download_drought_data:132 -

## Validación de la descarga de datos

### Función para validar los datos descargados

In [10]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            agriculture = SequiaSonoraSchema(**row.to_dict())
            validated_rows.append(agriculture.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [ ]:
# Get last generated file. Use max() to find the file with the latest modification time.
try:
  last_modified = max(
      (f for f in output_dir.iterdir() if f.is_file()),
      key=lambda f: f.stat().st_mtime,
  )
  print(f"Most recent file: {last_modified.name}")
  file_name = last_modified.name
except ValueError:
  print("The folder is empty or contains no files.")

Most recent file: sonora_sequia_2023.csv


In [ ]:
df_filtered_drought = pd.read_csv(str(output_dir)+"/" +file_name)
df_validated = validate_df(df_filtered_drought)
df_validated.head(5)

,CVE_CONCATENADA,CVE_ENT,CVE_MUN,fecha,categoria_sequia,Anio,severidad_num
0,26001,26,1,2023-01-15,D1,2023,2
1,26002,26,2,2023-01-15,D1,2023,2
2,26003,26,3,2023-01-15,D1,2023,2
3,26004,26,4,2023-01-15,D1,2023,2
4,26005,26,5,2023-01-15,D1,2023,2
